In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
suit_map = {'S': 0, 'H': 1, 'D': 2, 'C': 3}
rank_map = {'1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, 
            '8': 8, '9': 9, '10': 10, 'J': 11, 'Q': 12, 'K': 13}

def prepare_data(file_path):
    df = pd.read_csv(file_path, header=None)
    X_list = []
    y_list = []
    for _, row in df.iterrows():
        y_list.append(int(row[10]))
        ranks = []
        suits = []
        for i in range(0, 10, 2):
            suits.append(suit_map[str(row[i])])
            ranks.append(rank_map[str(row[i+1])])
        ranks.sort() 
        rank_counts = np.zeros(13)
        for r in ranks:
            rank_counts[r-1] += 1
        suit_counts = np.zeros(4)
        for s in suits:
            suit_counts[s] += 1
        features = np.concatenate([rank_counts, suit_counts, ranks])
        X_list.append(features)
    return torch.tensor(X_list, dtype=torch.float32), torch.tensor(y_list, dtype=torch.long)

X_train, y_train = prepare_data('train.csv')
X_test, y_test = prepare_data('test.csv')
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)

class PokerNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(22, 128),
            nn.ReLU(),
            nn.Dropout(0.2), 
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10) 
        )
    def forward(self, x):
        return self.net(x)

model = PokerNet()
criterion = nn.CrossEntropyLoss() 
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 50
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()           
        outputs = model(batch_X)         
        loss = criterion(outputs, batch_y) 
        loss.backward()                  
        optimizer.step()                 
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader):.4f}")

model.eval()
with torch.no_grad():
    test_outputs = model(X_test)
    _, predicted = torch.max(test_outputs, 1)
    accuracy = (predicted == y_test).sum().item() / y_test.size(0)
    print(f"最终测试集准确率: {accuracy * 100:.2f}%")

正在读取数据...


C:\Users\123\AppData\Local\Temp\ipykernel_22340\566248166.py:47: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  return torch.tensor(X_list, dtype=torch.float32), torch.tensor(y_list, dtype=torch.long)


开始训练...
Epoch [10/50], Loss: 0.1788
Epoch [20/50], Loss: 0.0912
Epoch [30/50], Loss: 0.0576
Epoch [40/50], Loss: 0.0481
Epoch [50/50], Loss: 0.0336

最终测试集准确率: 99.68%


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE, RandomOverSampler # 引入备用采样器
from collections import Counter

# ==========================================
# 1. 数据清洗与特征工程
# ==========================================

def process_poker_data(file_path):
    df = pd.read_csv(file_path, header=None)
    suit_map = {'S': 1, 'H': 2, 'D': 3, 'C': 4}
    rank_map = {'1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, 
                '8': 8, '9': 9, '10': 10, 'J': 11, 'Q': 12, 'K': 13}
    
    features = []
    labels = []
    
    for _, row in df.iterrows():
        try:
            label = int(row[10])
            cards = []
            valid = True
            for i in range(0, 10, 2):
                s, r = str(row[i]).strip(), str(row[i+1]).strip()
                if s not in suit_map or r not in rank_map:
                    valid = False; break
                cards.append((suit_map[s], rank_map[r]))
            
            if valid:
                # --- 特征提取逻辑 ---
                suits = [c[0] for c in cards]
                ranks = sorted([c[1] for c in cards])
                
                # 统计点数和花色频率
                rank_counts = np.bincount(ranks, minlength=14)[1:]
                suit_counts = np.bincount(suits, minlength=5)[1:]
                
                is_flush = 1 if max(suit_counts) == 5 else 0
                
                # 顺子逻辑
                u_ranks = sorted(list(set(ranks)))
                is_straight = 0
                if len(u_ranks) == 5:
                    if u_ranks[4] - u_ranks[0] == 4 or u_ranks == [1, 10, 11, 12, 13]:
                        is_straight = 1
                
                # 组合：点数统计(13) + 花色统计(4) + 是否同花(1) + 是否顺子(1) + 原始点数(5)
                f = list(rank_counts) + list(suit_counts) + [is_flush, is_straight] + ranks
                features.append(f)
                labels.append(label)
        except: continue
        
    return np.array(features), np.array(labels)

# ==========================================
# 2. 准备数据
# ==========================================
print("正在加载并提取特征...")
X_train, y_train = process_poker_data('train.csv')
X_test, y_test = process_poker_data('test.csv')

# 标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 处理不平衡 (修复后的 SMOTE 逻辑)
counts = Counter(y_train)
min_samples = min(counts.values())
k = min(5, min_samples - 1)

if k >= 1:
    print(f"使用 SMOTE 平衡数据 (k_neighbors={k})...")
    sampler = SMOTE(random_state=42, k_neighbors=k)
else:
    print("样本极少，使用随机采样平衡数据...")
    sampler = RandomOverSampler(random_state=42)

X_res, y_res = sampler.fit_resample(X_train_scaled, y_train)
y_res_cat = to_categorical(y_res, 10)
y_test_cat = to_categorical(y_test, 10)

# ==========================================
# 3. 建立 FCNN 模型
# ==========================================
print("正在构建模型...")
model = Sequential([
    Dense(256, input_dim=X_res.shape[1], activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])

# ==========================================
# 4. 训练与结果
# ==========================================
print("开始训练...")
model.fit(X_res, y_res_cat, validation_data=(X_test_scaled, y_test_cat), 
          epochs=30, batch_size=64, verbose=1)

print("\n--- 最终评估报告 ---")
loss, acc = model.evaluate(X_test_scaled, y_test_cat, verbose=0)
print(f"准确率: {acc*100:.2f}%")

y_pred = np.argmax(model.predict(X_test_scaled), axis=1)
print(classification_report(y_test, y_pred, digits=4))

正在加载并提取特征...
使用 SMOTE 平衡数据 (k_neighbors=3)...
正在构建模型...
开始训练...
Epoch 1/30


c:\ai\anaconda3\envs\ai\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1561/1561 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9288 - loss: 0.1892 - val_accuracy: 0.9916 - val_loss: 0.0445
Epoch 2/30
1561/1561 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9821 - loss: 0.0495 - val_accuracy: 0.9966 - val_loss: 0.0169
Epoch 3/30
1561/1561 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9879 - loss: 0.0339 - val_accuracy: 0.9978 - val_loss: 0.0119
Epoch 4/30
1561/1561 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9917 - loss: 0.0241 - val_accuracy: 0.9976 - val_loss: 0.0081
Epoch 5/30
1561/1561 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9935 - loss: 0.0189 - val_accuracy: 0.9976 - val_loss: 0.0126
Epoch 6/30
1561/1561 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9943 - loss: 0.0161 - val_accuracy: 0.9942 - val_loss: 0.0229
Epoch 7/30
1561/1561 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9950 - loss: 0.0146 - val_accuracy: 0.9978 - val_loss: 0.0089
Epoch 8/30
1561/1561 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9954 - loss: 0.0136 - val_accurac